In [ ]:
import sys
print(sys.version)

In [ ]:
import requests
import bs4
import pandas
import matplotlib
import seaborn
import scipy
import re
import time
import json

print("All libraries loaded successfully!")

In [ ]:
import requests

url = "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment&cityName=Navi-Mumbai"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)
print("Response size:", len(response.text), "characters")

In [ ]:
# Extract all property URLs first
urls = re.findall(r'"url":"(https://www\.magicbricks\.com/propertyDetails/.*?)"', response.text)

print(f"Total listings found: {len(urls)}")
print("\nFirst 3 URLs:")
for url in urls[:3]:
    print(url)

In [ ]:
# We'll store each listing as a dictionary
listings = []

for url in urls:
    # Extract the property details part from the URL
    # Example: "4-BHK-1619-Sq-ft-Multistorey-Apartment-FOR-Sale-Ghansoli-in-Navi-Mumbai"
    match = re.search(
        r'propertyDetails/(\d+)-BHK-(\d+)-Sq-ft-[\w-]+-FOR-Sale-([\w]+)-in-Navi-Mumbai',
        url
    )
    
    if match:
        bhk      = int(match.group(1))   # e.g. 4
        area     = int(match.group(2))   # e.g. 1619
        locality = match.group(3)        # e.g. Ghansoli
        
        listings.append({
            "bhk"      : bhk,
            "area_sqft": area,
            "locality" : locality
        })

print(f"Successfully parsed: {len(listings)} listings")
print("\nFirst 5:")
for l in listings[:5]:
    print(l)

In [ ]:
import pandas as pd

# Convert our list of dictionaries into a DataFrame
df = pd.DataFrame(listings)

# Let's see what we have
print(df.shape)   # rows, columns
print()
df.head(10)       # first 10 rows

In [ ]:
# Get a summary of the entire DataFrame
df.info()

In [ ]:
# Each listing object starts with "price": and has "priceD" nearby
# Let's extract all priceD values and their surrounding context

matches = re.findall(
    r'"devName":"(.*?)".*?"price":(\d+).*?"priceD":"(.*?)".*?"seoDesc":"(.*?)"',
    response.text
)

print(f"Listings with full data: {len(matches)}")
print("\nFirst 3:")
for m in matches[:3]:
    print({
        "developer": m[0],
        "price_raw": m[1],
        "price_fmt": m[2],
        "description": m[3]
    })

In [ ]:
def extract_locality(desc):
    # Pattern 1: "in Ghansoli, Navi Mumbai"
    m = re.search(r'in\s+([\w\s]+),\s+Navi Mumbai', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 2: "prime location of Panvel" or "location of Kharghar"
    m = re.search(r'location of\s+([\w\s]+?)[,\.]', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 3: "Panvel, Navi Mumbai has"
    m = re.search(r'^([\w\s]+),\s+Navi Mumbai\s+has', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 4: "premium Kharghar locality"
    m = re.search(r'premium\s+([\w\s]+?)\s+locality', desc)
    if m:
        return m.group(1).strip()

    # Pattern 5: "4 BHK flat in Panvel is"
    m = re.search(r'\d+\s+BHK\s+\w+\s+in\s+([\w\s]+?)\s+is', desc)
    if m:
        return m.group(1).strip()

    # Pattern 6: "apartment in Seawoods to"
    m = re.search(r'apartment in\s+([\w\s]+?)\s+to', desc)
    if m:
        return m.group(1).strip()

    return None

# Test again
for m in matches:
    desc = m[3]
    locality = extract_locality(desc)
    display = locality if locality else "NOT FOUND"
    print(f"{display:20} | {desc[:60]}")

In [ ]:
def parse_listing(block):
    """Extract all fields from one listing's JSON block"""
    
    # Price
    price_match = re.search(r'"price":(\d+)', block)
    price = int(price_match.group(1)) if price_match else None
    
    # Formatted price
    price_fmt_match = re.search(r'"priceD":"(.*?)"', block)
    price_fmt = price_fmt_match.group(1) if price_fmt_match else None
    
    # Developer
    dev_match = re.search(r'"devName":"(.*?)"', block)
    developer = dev_match.group(1) if dev_match else None
    
    # Description
    desc_match = re.search(r'"seoDesc":"(.*?)"', block)
    desc = desc_match.group(1) if desc_match else ""
    
    # BHK from description
    bhk_match = re.search(r'(\d+)\s+BHK', desc)
    bhk = int(bhk_match.group(1)) if bhk_match else None
    
    # Locality from description
    locality = extract_locality(desc)
    
    # Area sqft from description e.g. "carpet area 360 sqft"
    area_match = re.search(r'carpet area\s+(\d+)\s+sqft', desc)
    area = int(area_match.group(1)) if area_match else None
    
    # Project name from description e.g. "located in Siddhivinayak Apartment"
    project_match = re.search(r'located in\s+(.*?)[,\.]', desc)
    project = project_match.group(1).strip() if project_match else None
    
    # Possession date e.g. "dec '31" or "ready to move"
    possess_match = re.search(r'(ready to move|[\w]+\s+\'\d+)', desc, re.IGNORECASE)
    possession = possess_match.group(1).strip() if possess_match else None
    
    # Landmarks - extract by category code
    landmarks = re.findall(r'"(\d+)\|([^"]+)"', block)
    
    railway  = [name for code, name in landmarks if code == "19210"]
    hospital = [name for code, name in landmarks if code == "19203"]
    market   = [name for code, name in landmarks if code == "19206"]
    
    # Take first of each (nearest one)
    nearest_railway  = railway[0]  if railway  else None
    nearest_hospital = hospital[0] if hospital else None
    nearest_market   = market[0]   if market   else None
    
    return {
        "bhk"             : bhk,
        "price"           : price,
        "price_fmt"       : price_fmt,
        "locality"        : locality,
        "developer"       : developer,
        "project"         : project,
        "area_sqft"       : area,
        "possession"      : possession,
        "nearest_railway/metro_station" : nearest_railway,
        "nearest_hospital": nearest_hospital,
        "nearest_market"  : nearest_market
    }

print("Function defined successfully!")

In [ ]:
all_blocks = []

for page in range(1, 51):  # Pages 1 to 50
    
    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment&cityName=Navi-Mumbai&page={page}"
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        # Extract individual listing blocks
        # Each block starts with "devName" and ends after "seoDesc"
        blocks = re.findall(
            r'\{[^{}]*"devName":".*?"seoDesc":".*?"\}',
            response.text,
            re.DOTALL
        )
        all_blocks.extend(blocks)
        print(f"Page {page}: {len(blocks)} blocks (total: {len(all_blocks)})")
    else:
        print(f"Page {page}: Failed with status {response.status_code}")
    
    time.sleep(2)

print(f"\nDone! Total blocks collected: {len(all_blocks)}")

In [ ]:
parsed_listings = []

for block in all_blocks:
    listing = parse_listing(block)
    parsed_listings.append(listing)

df_rich = pd.DataFrame(parsed_listings)

print(f"Total rows: {len(df_rich)}")
print(f"\nMissing values per column:")
print(df_rich.isna().sum())
print(f"\nSample:")
print(df_rich.head(3).to_string())

In [ ]:
# Step 1: Drop rows where critical fields are missing
df_clean = df_rich.dropna(subset=["locality", "bhk", "price"])

print(f"After dropping missing locality/bhk/price: {len(df_clean)} rows")

# Step 2: Remove duplicate listings (same price + locality + bhk)
df_clean = df_clean.drop_duplicates(subset=["price", "locality", "bhk"])

print(f"After removing duplicates: {len(df_clean)} rows")

# Step 3: Convert bhk to integer (it's currently float because of the one missing value)
df_clean["bhk"] = df_clean["bhk"].astype(int)

# Step 4: Add price per sqft only where area is available
df_clean_with_area = df_clean.dropna(subset=["area_sqft"]).copy()
df_clean_with_area["price_per_sqft"] = (df_clean_with_area["price"] / df_clean_with_area["area_sqft"]).round(0)

print(f"Rows with area data for price/sqft analysis: {len(df_clean_with_area)}")

# Step 5: Save both versions
df_clean.to_csv("navi_mumbai_full.csv", index=False)
df_clean_with_area.to_csv("navi_mumbai_with_area.csv", index=False)

print("\nBoth files saved!")

In [ ]:
# Keep only localities with 10+ listings for reliable analysis
locality_counts = df_clean["locality"].value_counts()
valid_localities = locality_counts[locality_counts >= 10].index.tolist()

print(f"Localities with 10+ listings: {len(valid_localities)}")
print(valid_localities)

# Filter both DataFrames
df_main = df_clean[df_clean["locality"].isin(valid_localities)].copy()
df_area = df_clean_with_area[df_clean_with_area["locality"].isin(valid_localities)].copy()

print(f"\nRows in main dataset: {len(df_main)}")
print(f"Rows with area data: {len(df_area)}")

In [ ]:
# Average price per sqft by locality - sorted cheapest first
price_analysis = df_area.groupby("locality")["price_per_sqft"].agg(["mean", "median", "count"]).round(0)
price_analysis.columns = ["avg_price_sqft", "median_price_sqft", "listings"]
price_analysis = price_analysis.sort_values("median_price_sqft")

print("Price per Sq Ft by Locality (cheapest first):")
print()
print(f"{'Locality':<20} {'Avg ₹/sqft':>12} {'Median ₹/sqft':>15} {'Listings':>10}")
print("-" * 60)
for locality, row in price_analysis.iterrows():
    print(f"{locality:<20} {row['avg_price_sqft']:>12,.0f} {row['median_price_sqft']:>15,.0f} {row['listings']:>10.0f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Sort by median price for clean visualization
plot_data = price_analysis.reset_index().sort_values("median_price_sqft")

plt.figure(figsize=(12, 7))

# Main bars - median price
bars = plt.barh(plot_data["locality"], plot_data["median_price_sqft"], 
                color="steelblue", alpha=0.8, label="Median")

# Overlay dots - average price
plt.scatter(plot_data["avg_price_sqft"], plot_data["locality"], 
            color="orange", zorder=5, s=80, label="Average")

# Add listing count as text on each bar
for i, (_, row) in enumerate(plot_data.iterrows()):
    plt.text(500, i, f"n={row['listings']:.0f}", 
             va="center", fontsize=8, color="white", fontweight="bold")

# Reference lines
plt.axvline(x=10000, color="green", linestyle="--", alpha=0.7, label="₹10,000")
plt.axvline(x=20000, color="red", linestyle="--", alpha=0.7, label="₹20,000")

plt.xlabel("Price per Sq Ft (₹)")
plt.title("Navi Mumbai — Price per Sq Ft by Locality\n(bars = median, dots = average)", 
          fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average price by locality AND BHK type
bhk_analysis = df_main[df_main["bhk"].isin([1, 2, 3])].groupby(
    ["locality", "bhk"]
)["price"].median().round(0) / 100000  # Convert to Lakhs

# Reshape into a table: localities as rows, BHK as columns
bhk_table = bhk_analysis.unstack("bhk")
bhk_table.columns = ["1 BHK", "2 BHK", "3 BHK"]

# Sort by 2 BHK price
bhk_table = bhk_table.sort_values("2 BHK")

print("Median Price by Locality and BHK (in Lakhs):")
print()
print(f"{'Locality':<20} {'1 BHK':>10} {'2 BHK':>10} {'3 BHK':>10}")
print("-" * 52)
for locality, row in bhk_table.iterrows():
    bhk1 = f"₹{row['1 BHK']:.0f}L" if not pd.isna(row['1 BHK']) else "N/A"
    bhk2 = f"₹{row['2 BHK']:.0f}L" if not pd.isna(row['2 BHK']) else "N/A"
    bhk3 = f"₹{row['3 BHK']:.0f}L" if not pd.isna(row['3 BHK']) else "N/A"
    print(f"{locality:<20} {bhk1:>10} {bhk2:>10} {bhk3:>10}")

In [ ]:
plt.figure(figsize=(10, 8))

sns.heatmap(
    bhk_table,
    annot=True,           # Show numbers inside cells
    fmt=".0f",            # No decimal places
    cmap="RdYlGn_r",      # Red = expensive, Green = affordable
    linewidths=0.5,       # Grid lines between cells
    cbar_kws={"label": "Price in Lakhs"}
)

plt.title("Navi Mumbai — Median Flat Price by Locality & BHK (₹ Lakhs)", 
          fontsize=13, pad=15)
plt.xlabel("Flat Type")
plt.ylabel("Locality")
plt.tight_layout()
plt.show()

In [ ]:
mumbai_url = "https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment&cityName=Mumbai"

response_mumbai = requests.get(mumbai_url, headers=headers)

print("Status code:", response_mumbai.status_code)
print("Response size:", len(response_mumbai.text), "characters")

# Check if same block structure exists
test_blocks = re.findall(
    r'\{[^{}]*"devName":".*?"seoDesc":".*?"\}',
    response_mumbai.text,
    re.DOTALL
)
print(f"Blocks found on page 1: {len(test_blocks)}")

In [ ]:
all_blocks_combined = []

cities = {
    "Navi-Mumbai": "Navi Mumbai",
    "Mumbai": "Mumbai"
}

for city_param, city_name in cities.items():
    print(f"\nScraping {city_name}...")
    
    for page in range(1, 51):
        url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment&cityName={city_param}&page={page}"
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            blocks = re.findall(
                r'\{[^{}]*"devName":".*?"seoDesc":".*?"\}',
                response.text,
                re.DOTALL
            )
            
            # Tag each block with city before storing
            for block in blocks:
                all_blocks_combined.append({
                    "city" : city_name,
                    "block": block
                })
            
            print(f"  Page {page}: {len(blocks)} blocks (total: {len(all_blocks_combined)})")
        else:
            print(f"  Page {page}: Failed with status {response.status_code}")
        
        time.sleep(2)

print(f"\nDone! Total blocks: {len(all_blocks_combined)}")

In [ ]:
def extract_locality(desc):
    # Pattern 1: "in Ghansoli, Navi Mumbai" OR "in Bandra East, Mumbai"
    m = re.search(r'in\s+([\w\s]+),\s+(Navi Mumbai|Mumbai)', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 2: "prime location of Panvel" or "area of Mahalakshmi"
    m = re.search(r'(?:location|area) of\s+([\w\s]+?)[,\.]', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 3: "Panvel, Navi Mumbai has" OR "Borivali East, Mumbai has"
    m = re.search(r'^([\w\s]+),\s+(?:Navi Mumbai|Mumbai)\s+has', desc)
    if m:
        return m.group(1).strip()
    
    # Pattern 4: "premium Kharghar locality"
    m = re.search(r'premium\s+([\w\s]+?)\s+locality', desc)
    if m:
        return m.group(1).strip()

    # Pattern 5: "4 BHK flat in Panvel is" OR "4 BHK apartment in Kandivali West"
    m = re.search(r'\d+\s+BHK\s+\w+\s+(?:is\s+)?(?:immediately\s+)?(?:available\s+)?(?:for\s+sale\s+)?in\s+([\w\s]+?)(?:\.|,|\s+is|\s+has|\s+This)', desc)
    if m:
        return m.group(1).strip()

    # Pattern 6: "apartment in Seawoods to"
    m = re.search(r'(?:flat|apartment|residence)\s+in\s+([\w\s]+?)\s+to', desc)
    if m:
        return m.group(1).strip()

    # Pattern 7: "for sale in Kandivali West.This"
    m = re.search(r'for sale in\s+([\w\s]+?)[\.,$]', desc)
    if m:
        return m.group(1).strip()

    # Pattern 8: "Now available for sale: this 3 BHK apartment in Goregaon West, Mumbai"
    m = re.search(r'this\s+\d+\s+BHK\s+\w+\s+in\s+([\w\s]+?)(?:,|\.|$)', desc)
    if m:
        return m.group(1).strip()

    return None

print("Function updated!")

In [ ]:
all_listings = []

for item in all_blocks_combined:
    city  = item["city"]
    block = item["block"]
    
    listing = parse_listing(block)
    listing["city"] = city
    
    all_listings.append(listing)

df_combined = pd.DataFrame(all_listings)

# Clean - drop missing critical fields
df_combined = df_combined.dropna(subset=["locality", "bhk", "price"])

# Remove duplicates
df_combined = df_combined.drop_duplicates(subset=["price", "locality", "bhk"])

# Convert bhk to int
df_combined["bhk"] = df_combined["bhk"].astype(int)

print(f"Total clean listings: {len(df_combined)}")
print(f"\nBy city:")
print(df_combined["city"].value_counts())
print(f"\nTop 10 localities overall:")
print(df_combined["locality"].value_counts().head(10))

In [ ]:
# Save master dataset
df_combined.to_csv("mumbai_realestate_master.csv", index=False)

# Add price per sqft where area is available
df_with_area = df_combined.dropna(subset=["area_sqft"]).copy()
df_with_area["price_per_sqft"] = (df_with_area["price"] / df_with_area["area_sqft"]).round(0)

print(f"Master dataset saved: {len(df_combined)} listings")
print(f"Listings with area data: {len(df_with_area)}")
print(f"\nColumns in dataset:")
for col in df_combined.columns:
    non_null = df_combined[col].notna().sum()
    pct = (non_null / len(df_combined) * 100).round(1)
    print(f"  {col:<20} {non_null:>5} non-null ({pct}%)")

## Part 2 — Data Enrichment & Price Prediction

### Step 1: Clean Possession Data

In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\KL\Real estate\mumbai_realestate_master.csv")

# Standardize possession values
df["possession_clean"] = df["possession"].str.strip().str.lower()

def categorize_possession(val):
    if pd.isna(val):
        return "Unknown"
    val = str(val).strip().lower()
    if val == "ready to move":
        return "Ready to Move"
    else:
        return "Under Construction"

df["possession_status"] = df["possession_clean"].apply(categorize_possession)

print("Possession status distribution:")
print(df["possession_status"].value_counts())

### Step 2: Remove Bad Localities

In [ ]:
# Some area values (e.g. "288 sqft") got picked up as localities during scraping
# Remove these bad rows

bad = df[df["locality"].str.contains("sqft", case=False, na=False)]["locality"].unique()
print(f"Bad localities found: {len(bad)}")

df_clean = df[~df["locality"].str.contains("sqft", case=False, na=False)].copy()

print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} rows")

df = df_clean.copy()

### Step 3: Save Updated Dataset

In [ ]:
df.to_csv(r"C:\Users\KL\Real estate\mumbai_realestate_master.csv", index=False)
print(f"Saved! Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

### Step 4: Build Price Prediction Model

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Use only rows with all required features
model_data = df.dropna(subset=["bhk", "area_sqft", "locality", "city", "price"]).copy()

# Remove extreme outliers above 99th percentile
p99 = model_data["price"].quantile(0.99)
model_data = model_data[model_data["price"] <= p99]

print(f"Training rows after outlier removal: {len(model_data)}")

# Encode locality and city as numbers
# Random Forest needs numbers, not text
le_locality = LabelEncoder()
le_city = LabelEncoder()

model_data["locality_encoded"] = le_locality.fit_transform(model_data["locality"])
model_data["city_encoded"]     = le_city.fit_transform(model_data["city"])

X = model_data[["bhk", "area_sqft", "locality_encoded", "city_encoded"]]
y = model_data["price"]

print(f"Features: {X.columns.tolist()}")

### Step 5: Train and Evaluate

In [ ]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest — 100 decision trees averaged together
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate on unseen test data
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"R2 Score: {r2:.3f}  (1.0 = perfect, 0.0 = useless)")
print(f"Mean Absolute Error: Rs {mae/100000:.1f} Lakhs")

# Feature importance — what drives price most?
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance:")
for feat, imp in importances.items():
    print(f"  {feat}: {imp:.3f}")

### Step 6: Save Model and Encoders

In [ ]:
import pickle

with open(r"C:\Users\KL\Real estate\price_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(r"C:\Users\KL\Real estate\le_locality.pkl", "wb") as f:
    pickle.dump(le_locality, f)

with open(r"C:\Users\KL\Real estate\le_city.pkl", "wb") as f:
    pickle.dump(le_city, f)

print("Model and encoders saved!")

# Test prediction — 2 BHK 900 sqft in Kharghar
test_input = pd.DataFrame(
    [[2, 900, le_locality.transform(["Kharghar"])[0], le_city.transform(["Navi Mumbai"])[0]]],
    columns=["bhk", "area_sqft", "locality_encoded", "city_encoded"]
)
predicted = model.predict(test_input)[0]
print(f"Test: 2 BHK 900 sqft Kharghar -> Rs {predicted/100000:.1f}L")